# Extract Custom Fields from Your File

This notebook demonstrates how to use analyzers to extract custom fields from your input files.

Source: https://github.com/Azure-Samples/azure-ai-content-understanding-python.git
further expanded for training purposes.

In [ ]:
import logging
import json
import os
import sys
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
import uuid # for generating unique analyzer IDs

## Analyzer Templates

Below is a collection of analyzer templates designed to extract fields from various input file types.

These templates are highly customizable, allowing you to modify them to suit your specific needs. For additional verified templates from Microsoft, please visit [here](https://github.com/Azure-Samples/azure-ai-content-understanding-python/tree/fb251ace83a98c9c82b9871f217c88d73e676bb7/analyzer_templates).

In [ ]:
analyzer_template_folder = "analyzer_templates_NEW"
data_folder = "assets"

extraction_templates = {
    # Extract fields from invoices (no grounding sources or confidence scores).
    "invoice": (f'{analyzer_template_folder}/invoice.json', f'{data_folder}/docs/invoice.pdf'),

    # Extract fields from invoices, including grounding sources and confidence scores (optional add-on).
    "invoice_field_source": (f'{analyzer_template_folder}/invoice_field_source.json', f'{data_folder}/docs/invoice.pdf'),

    # Analyze a photo to extract a short description and some boolean attributes.
    "university_photo": (f'{analyzer_template_folder}/university_photo_analyzer.json', f'{data_folder}/images/outdoor-students.jpg'),

    # Extract insights from an image representing a chart (e.g., bar chart, pie chart).
    "chart": (f'{analyzer_template_folder}/image_chart.json', f'{data_folder}/charts/figure_1.png'),

    # Extract insights from call recordings (e.g., summary, topics, mentioned companies, and people).
    "call_recording": (f'{analyzer_template_folder}/call_recording_analytics.json', f'{data_folder}/audio/callCenterRecording.mp3'),

    # Extract summary and sentiment from conversation audio (e.g., customer service calls).
    "conversation_audio": (f'{analyzer_template_folder}/conversational_audio_analytics.json', f'{data_folder}/audio/callCenterRecording.mp3'),

    # Extract descriptions and sentiment analysis from marketing videos.
    "marketing_video": (f'{analyzer_template_folder}/marketing_video.json', f'{data_folder}/video/FlightSimulator.mp4'),
}

## Create Azure AI Content Understanding Client

> The [AzureContentUnderstandingClient](../python/content_understanding_client.py) is a utility class containing functions to interact with the Content Understanding API. Before the official release of the Content Understanding SDK, it can be regarded as a lightweight SDK.


In [58]:
load_dotenv(override=True, dotenv_path=find_dotenv())

True

In [59]:
logging.basicConfig(level=logging.INFO)

In [60]:
AZURE_AI_ENDPOINT = os.getenv("AZURE_CU_ENDPOINT_NEW")
AZURE_AI_API_VERSION =  os.getenv("AZURE_CU_API_VERSION_NEW", "2025-05-01-preview")
print(f"Current Azure Content Understanding endpoint: {AZURE_AI_ENDPOINT}")
print(f"Current Azure Content Understanding API version: {AZURE_AI_API_VERSION}")

Current Azure Content Understanding endpoint: https://epaifhub6672084982.cognitiveservices.azure.com/
Current Azure Content Understanding API version: 2025-05-01-preview


In [61]:
# only if necessary, add the parent directory to the path to use shared modules
# parent_dir = Path(Path.cwd()).parent
# sys.path.append(str(parent_dir))

# import the utility class AzureContentUnderstandingClient, which is a wrapper around the Azure Content Understanding REST API client
from python.content_understanding_client_NEW import AzureContentUnderstandingClient

In [62]:
credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")

INFO:azure.identity._credentials.environment:No environment configuration found.
INFO:azure.identity._credentials.managed_identity:ManagedIdentityCredential will use IMDS


In [114]:
client = AzureContentUnderstandingClient(
    endpoint=AZURE_AI_ENDPOINT,
    api_version=AZURE_AI_API_VERSION,
    token_provider=token_provider,
    # x_ms_useragent="azure-ai-content-understanding-python/field_extraction", # This header is used for sample usage telemetry, please comment out this line if you want to opt out.
)

INFO:azure.identity._internal.decorators:AzureCliCredential.get_token_info succeeded
INFO:azure.identity._credentials.default:DefaultAzureCredential acquired a token from AzureCliCredential


## Create Analyzer from the Template

Specify the analyzer template you want to use and provide a name for the analyzer to be created based on the template.

In [116]:
#extract keys from extraction_templates
analyzer_names = list(extraction_templates.keys())
print(f"Available analyzers: {analyzer_names}")

Available analyzers: ['invoice', 'invoice_field_source', 'university_photo', 'chart', 'call_recording', 'conversation_audio', 'marketing_video']


In [117]:
ANALYZER_TEMPLATE = "chart"

(analyzer_template_path, analyzer_sample_file_path) = extraction_templates[ANALYZER_TEMPLATE]
print(f"Using analyzer template: {analyzer_template_path}")
print(f"Using sample file: {analyzer_sample_file_path}")

Using analyzer template: analyzer_templates_NEW/image_chart.json
Using sample file: assets/charts/figure_1.png


In [118]:
CUSTOM_ANALYZER_ID = "field-extraction-sample-" + str(uuid.uuid4())
response = client.begin_create_analyzer(CUSTOM_ANALYZER_ID, analyzer_template_path=analyzer_template_path)
result = client.poll_result(response)

print(json.dumps(result, indent=2))

INFO:python.content_understanding_client_NEW:Analyzer field-extraction-sample-b04d0a67-9e6a-4b07-9444-76b6730fb7bd create request accepted.
INFO:python.content_understanding_client_NEW:Request result is ready after 0.00 seconds.


{
  "id": "03d10bba-a6a1-4d78-ac0d-2991ec93c733",
  "status": "Succeeded",
  "result": {
    "analyzerId": "field-extraction-sample-b04d0a67-9e6a-4b07-9444-76b6730fb7bd",
    "description": "Extract detailed structured information from charts and diagrams.",
    "createdAt": "2025-06-16T12:16:34Z",
    "lastModifiedAt": "2025-06-16T12:16:34Z",
    "baseAnalyzerId": "prebuilt-imageAnalyzer",
    "config": {
      "returnDetails": true,
      "enableOcr": false,
      "disableContentFiltering": false
    },
    "fieldSchema": {
      "fields": {
        "Title": {
          "type": "string",
          "method": "generate",
          "description": "Verbatim title of the chart."
        },
        "ChartType": {
          "type": "string",
          "method": "classify",
          "description": "The type of chart.",
          "enum": [
            "area",
            "bar",
            "box",
            "bubble",
            "candlestick",
            "funnel",
            "heatmap",
  

## Extract Fields Using the Analyzer

After the analyzer is successfully created, we can use it to analyze our input files.

In [119]:
response = client.begin_analyze(CUSTOM_ANALYZER_ID, file_location=analyzer_sample_file_path)
result_json = client.poll_result(response, timeout_seconds=240)

print(json.dumps(result_json, indent=2))

INFO:python.content_understanding_client_NEW:Analyzing file assets/charts/figure_1.png with analyzer: field-extraction-sample-b04d0a67-9e6a-4b07-9444-76b6730fb7bd
INFO:python.content_understanding_client_NEW:Request eef2ded3-cc11-43f5-9fd3-32b32c6da21c in progress ...
INFO:python.content_understanding_client_NEW:Request eef2ded3-cc11-43f5-9fd3-32b32c6da21c in progress ...
INFO:python.content_understanding_client_NEW:Request result is ready after 5.04 seconds.


{
  "id": "eef2ded3-cc11-43f5-9fd3-32b32c6da21c",
  "status": "Succeeded",
  "result": {
    "analyzerId": "field-extraction-sample-b04d0a67-9e6a-4b07-9444-76b6730fb7bd",
    "apiVersion": "2025-05-01-preview",
    "createdAt": "2025-06-16T12:16:48Z",
    "warnings": [],
    "contents": [
      {
        "markdown": "![image](image)\n",
        "fields": {
          "Title": {
            "type": "string",
            "valueString": "2023 Real GDP"
          },
          "ChartType": {
            "type": "string",
            "valueString": "bar"
          },
          "TopicKeywords": {
            "type": "array",
            "valueArray": [
              {
                "type": "string",
                "valueString": "GDP"
              },
              {
                "type": "string",
                "valueString": "economy"
              },
              {
                "type": "string",
                "valueString": "2023"
              },
              {
              

In [120]:
# save the result to a json file
output_dir = "results/field_extraction"
Path(output_dir).mkdir(parents=True, exist_ok=True)
file_name = Path(analyzer_sample_file_path).name
output_file = f"{output_dir}/{ANALYZER_TEMPLATE}_{file_name}.json"

with open(output_file, "w") as f:
    json.dump(result_json, f, indent=2)

## Clean Up
Optionally, delete the sample analyzer from your resource. In typical usage scenarios, you would analyze multiple files using the same analyzer.

In [121]:
client.delete_analyzer(CUSTOM_ANALYZER_ID)

INFO:python.content_understanding_client_NEW:Analyzer field-extraction-sample-b04d0a67-9e6a-4b07-9444-76b6730fb7bd deleted.


<Response [204]>